In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

print("All imports successful!")

All imports successful!


In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

df = pd.read_csv("../data/processed/retail_inventory_cleaned.csv")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["Units Sold"] = pd.to_numeric(df["Units Sold"], errors="coerce")

df = df.dropna(
    subset=["Date", "Price", "Units Sold", "Product ID", "Store ID"]
).copy()

df = df[df["Price"] > 0].copy()

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
display(df.head())

Dataset shape: (73100, 14)
Date range: 2022-01-01 00:00:00 to 2024-01-01 00:00:00


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Year,Month,DayOfWeek,Revenue
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,2022,1,5,4254.50
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,2022,1,5,9451.50
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,2022,1,5,1819.35
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,2022,1,5,1995.92
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,2022,1,5,1030.96


In [5]:
pair_summary = df.groupby(
    ["Product ID", "Store ID"]
).agg(
    observations=("Price", "size"),
    unique_prices=("Price", "nunique"),
    min_price=("Price", "min"),
    max_price=("Price", "max")
)

pair_summary["price_range"] = (
    pair_summary["max_price"] - pair_summary["min_price"]
)

display(pair_summary.head(10))

print("Total product-store pairs:", len(pair_summary))
print(
    "Pairs with changing prices:",
    (pair_summary["unique_prices"] > 1).sum()
)
print(
    "Pairs with constant prices:",
    (pair_summary["unique_prices"] == 1).sum()
)

observations  unique_prices  min_price  max_price  \
Product ID Store ID                                                      
P0001      S001               731            697      10.26      99.97   
           S002               731            694      10.81      99.99   
           S003               731            693      10.19      99.86   
           S004               731            694      10.29      99.88   
           S005               731            709      10.07      99.76   
P0002      S001               731            697      10.15      99.95   
           S002               731            706      10.01      99.99   
           S003               731            715      10.16      99.67   
           S004               731            706      10.60      99.96   
           S005               731            705      10.08      99.99   

                     price_range  
Product ID Store ID               
P0001      S001            89.71  
           S002            89.18  
           S003            89.67  
           S004            89.59  
           S005            89.69  
P0002      S001            89.80  
           S002            89.98  
           S003            89.51  
           S004            89.36  
           S005            89.91

Total product-store pairs: 100
Pairs with changing prices: 100
Pairs with constant prices: 0


In [6]:
simple_model = smf.ols(
    'Q("Units Sold") ~ Price',
    data=df
).fit(cov_type="HC3")

print(simple_model.summary())


                            OLS Regression Results                            
Dep. Variable:        Q("Units Sold")   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                   0.08569
Date:                Sat, 17 Oct 2026   Prob (F-statistic):              0.770
Time:                        21:39:45   Log-Likelihood:            -4.4661e+05
No. Observations:               73100   AIC:                         8.932e+05
Df Residuals:                   73098   BIC:                         8.932e+05
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    136.2153      0.942    144.594      0.0

In [7]:
print("Price coefficient:", simple_model.params["Price"])
print("P-value:", simple_model.pvalues["Price"])

Price coefficient: 0.004527257865186157
P-value: 0.7697287969362305


In [8]:
controlled_model = smf.ols(
    'Q("Units Sold") ~ Price + C(Q("Product ID")) '
    '+ C(Q("Store ID")) + C(Date)',
    data=df
).fit(cov_type="HC3")

print(controlled_model.summary())

                            OLS Regression Results                            
Dep. Variable:        Q("Units Sold")   R-squared:                       0.009
Model:                            OLS   Adj. R-squared:                 -0.001
Method:                 Least Squares   F-statistic:                    0.9309
Date:                Sat, 17 Oct 2026   Prob (F-statistic):              0.912
Time:                        21:42:09   Log-Likelihood:            -4.4626e+05
No. Observations:               73100   AIC:                         8.940e+05
Df Residuals:                   72345   BIC:                         9.010e+05
Df Model:                         754                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [9]:
comparison = pd.DataFrame({
    "Model": ["Simple regression", "Product/store/date controls"],
    "Price coefficient": [
        simple_model.params["Price"],
        controlled_model.params["Price"]
    ],
    "P-value": [
        simple_model.pvalues["Price"],
        controlled_model.pvalues["Price"]
    ]
})

display(comparison.round(4))

,Model,Price coefficient,P-value
0,Simple regression,0.0045,0.7697
1,Product/store/date controls,0.0045,0.7736
